# Lab 4 — Build, Tool, Evaluate
## Agentic Integration for Precision Recruiting

**Mission:** Build a bounded agent that turns two recruiters and 16 field hours into a proposed plan for human review. It may use approved tools, but it may not invent evidence, exceed the budget, contact a school, or operationalize its own plan.

**Estimated time:** 75 minutes

All schools, records, policies, and outcomes are fictional classroom content.

> **Use your coding assistant as a teammate.** Give it the current cell, the self-check output, and the goal. Ask it to explain the smallest useful change rather than rewriting the notebook.

Suggested prompt:

> I am working in a classroom Jupyter notebook. Explain what this self-check is testing, then suggest the smallest edit to the marked variables. Do not change the data or the test.

> **Completed instructor version.** Exercise values and functions are filled in; self-checks demonstrate the intended behavior.


In [ ]:
#@title
from IPython.display import HTML, display
display(HTML("<div style='background-color:rgba(128,128,128,.12);border:1px solid rgba(128,128,128,.28);border-radius:6px;padding:12px 16px;margin:8px 0 12px'><h3 style='margin:0 0 6px'>0.2 Configure the OpenAI Agents SDK</h3><p style='margin:0'>This lab defines OpenAI Agents SDK agents, then uses <code>Runner</code> for repeated trials and tool use. Use a temporary workshop key; clear it and generated outputs before sharing.</p></div>"))

# Uncomment once if needed:
# %pip install -q openai-agents
import os
from typing import Literal
from pydantic import BaseModel, Field
from agents import Agent, Runner, function_tool

MODEL = "gpt-5.4-mini"
OPENAI_API_KEY = ""  # Paste the temporary workshop key here.
if not OPENAI_API_KEY.strip():
    raise ValueError("Paste a temporary workshop key into OPENAI_API_KEY before running this lab.")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY.strip()


In [ ]:
from IPython.display import display, Markdown, HTML
from pathlib import Path
import json, re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def check(name, condition, hint=""):
    try: passed = bool(condition)
    except Exception as exc: passed, hint = False, f"{hint} ({type(exc).__name__}: {exc})"
    print(("✅" if passed else "❌") + f" {name}")
    if not passed and hint: print(f"   Hint: {hint}")
    return passed

def section_card(number, title, text):
    display(HTML(f"<div style='background:#eef5fb;border-left:6px solid #1f5a91;padding:12px 16px;margin:10px 0'><h2 style='margin:0'>{number}. {title}</h2><p style='margin:6px 0 0'>{text}</p></div>"))

def find_data(name):
    for path in (Path('../data') / name, Path('data') / name):
        if path.exists(): return path
    raise FileNotFoundError(name)

HOURS_AVAILABLE = 16
HUMAN_APPROVAL_REQUIRED = True


# Section 1 — Build a Basic Agent

**Return to the presentation:** *Lab 4 Section 1 — making a basic agent.*

An agent has a goal, instructions, state, allowed actions, and a stop condition. Before it has tools, it can sound helpful but cannot establish a trustworthy recruiting plan.

In [ ]:
section_card('1', 'Build the bounded basic agent', 'Start with a realistic mission and raw records. Your task is to define the agent that responds appropriately before it has any tools.')
raw_events = pd.read_csv(find_data('raw_recruiting_events.csv')).head(12)
MISSION = 'Two recruiters have 16 field hours next week. Propose where to focus, what engagement to run, and what recruiters should know before acting.'
raw_packet = raw_events.to_csv(index=False)
print(MISSION)
display(raw_events)


## 1.1 Define the agent contract

Improve the agent's instructions rather than chaining unrelated prompts. Its no-tool actions are `respond_with_provisional_plan`, `request_evidence`, and `escalate`. The goal is controlled, inspectable behavior—not a final decision.

In [ ]:
class BasicAgentDecision(BaseModel):
    status: Literal['provisional', 'escalate']
    proposed_hours: int | None = Field(description='Hours in the preliminary proposal, if any.')
    assumptions: list[str] = Field(description='Assumptions distinct from supplied evidence.')
    evidence_used: list[str] = Field(description='Only evidence explicitly supplied in the request.')
    next_step: str = Field(description='A concise next step or uncertainty to resolve.')
    external_action_taken: bool = Field(description='Must be false; this agent cannot contact or schedule anyone.')

# TODO: Write instructions for a bounded preliminary-planning agent. Require it to
# respect the 16-hour budget, separate assumptions from raw evidence, and never
# claim that it contacted a school or executed a plan.
CONTRACT_INSTRUCTIONS = """You are a bounded preliminary-planning agent. You may propose a preliminary plan from the supplied raw records, but it is not an approved recommendation. Keep any proposed allocation at or below 16 hours. Separate assumptions from supplied evidence. Do not invent local hosting, benefits, or policy facts. Set external_action_taken to false: you cannot contact schools, schedule an event, or execute a plan. State one next step or uncertainty that a human should resolve."""

bounded_basic_agent = Agent(name='Bounded Precision Recruiting Agent', model=MODEL, output_type=BasicAgentDecision, instructions=CONTRACT_INSTRUCTIONS)

async def run_bounded_trial():
    return (await Runner.run(bounded_basic_agent, f'MISSION\n{MISSION}\n\nRAW RECORDS\n{raw_packet}')).final_output

bounded_trials = [await run_bounded_trial() for _ in range(3)]
display(pd.DataFrame([trial.model_dump() for trial in bounded_trials]))


In [ ]:
check('Every live trial stays within the 16-hour budget', all(t.proposed_hours is None or t.proposed_hours <= HOURS_AVAILABLE for t in bounded_trials))
check('No live trial claims an external action was taken', all(not t.external_action_taken for t in bounded_trials))
check('Every live trial separates assumptions from supplied evidence', all(isinstance(t.assumptions, list) and isinstance(t.evidence_used, list) for t in bounded_trials))


**Pause and return to the presentation.** Better instructions can make outputs more stable and inspectable, but they cannot supply the ranking calculation, action evidence, or approved local facts.

# Section 2 — Add Tools to the Agent

**Return to the presentation:** *Lab 4 Section 2 — adding tools.*

Now the agent can select from narrow, approved functions. The recommender answers **where** and **what**; RAG provides grounded preparation information.

In [ ]:
section_card('2', 'Load the earlier lab capabilities', 'These functions use the validated Lab 2 artifacts and the Lab 3 approved corpus.')
schools = pd.read_csv(find_data('school_summary.csv'))
events = pd.read_csv(find_data('clean_recruiting_events.csv'))
school_profiles = pd.read_csv(data_url('school_profiles.csv')).set_index('school_name')
action_profiles = pd.read_csv(data_url('action_profiles.csv')).set_index('action')

def rank_schools(top_k=3):
    ranked = schools.copy()
    ranked['qualified_rate'] = ranked['qualified'] / ranked['appointments']
    ranked['contracts_per_hour'] = ranked['contracts'] / ranked['recruiter_hours']
    for column in ['contracts_per_hour', 'qualified_rate', 'access_score']:
        span = ranked[column].max() - ranked[column].min()
        ranked[f'{column}_norm'] = (ranked[column] - ranked[column].min()) / span if span else 0
    ranked['opportunity_score'] = .60 * ranked['contracts_per_hour_norm'] + .25 * ranked['qualified_rate_norm'] + .15 * ranked['access_score_norm']
    ranked = ranked.query('distance_miles <= 30 and historical_events >= 4').sort_values('opportunity_score', ascending=False).head(top_k)
    return ranked[['school_name','opportunity_score','historical_events','distance_miles']].rename(columns={'school_name':'school'}).to_dict('records')


corpus = []
for path in sorted(find_data('rag_corpus').glob('*.md')):
    text = path.read_text(encoding='utf-8')
    source = re.search(r'source_id:\s*(.+)', text).group(1).strip()
    corpus.append({'source_id':source, 'text':text})
vectorizer = TfidfVectorizer(stop_words='english')
corpus_matrix = vectorizer.fit_transform([x['text'] for x in corpus])
def retrieve_brief_evidence(school, action, question):
    scores = cosine_similarity(vectorizer.transform([question]), corpus_matrix)[0]
    picks = scores.argsort()[::-1][:3]
    return [{'source_id':corpus[i]['source_id'], 'text':corpus[i]['text'][:500]} for i in picks]

rank_schools()


In [ ]:
# Load the Lab 2 profile artifacts here so the tool definition is self-contained.
school_profiles = pd.read_csv(data_url('school_profiles.csv')).set_index('school_name')
action_profiles = pd.read_csv(data_url('action_profiles.csv')).set_index('action')

@function_tool
def rank_schools(top_k: int = 3) -> list[dict]:
    """Return feasible schools ranked by the Lab 2 opportunity score."""
    ranked = schools.copy(); ranked['qualified_rate']=ranked['qualified']/ranked['appointments']; ranked['contracts_per_hour']=ranked['contracts']/ranked['recruiter_hours']
    for col in ['contracts_per_hour','qualified_rate','access_score']:
        span=ranked[col].max()-ranked[col].min(); ranked[col+'_norm']=(ranked[col]-ranked[col].min())/span if span else 0
    ranked['opportunity_score']=.60*ranked['contracts_per_hour_norm']+.25*ranked['qualified_rate_norm']+.15*ranked['access_score_norm']
    return ranked.query('distance_miles <= 30 and historical_events >= 4').nlargest(top_k,'opportunity_score')[['school_name','opportunity_score','historical_events']].rename(columns={'school_name':'school'}).to_dict('records')

@function_tool
def recommend_action(school: str) -> dict:
    """Compute a Lab 2-style hybrid action ranking from event and profile artifacts."""
    if school not in school_profiles.index:
        return {'status': 'no_school_profile', 'school': school}
    dimensions = ['cyber', 'engineering', 'mechanical', 'healthcare', 'education']
    profile = school_profiles.loc[[school], dimensions]
    content_scores = cosine_similarity(profile, action_profiles[dimensions])[0]
    content = pd.Series(content_scores, index=action_profiles.index)
    action_history = events.groupby('action').agg(contracts=('contracts','sum'), hours=('recruiter_hours','sum'))
    action_history['outcome_per_hour'] = action_history['contracts'] / action_history['hours']
    historical = action_history['outcome_per_hour'].reindex(action_profiles.index).fillna(0)
    historical_norm = (historical - historical.min()) / (historical.max() - historical.min())
    ranking = pd.DataFrame({'content_score': content, 'historical_norm': historical_norm})
    ranking['hybrid_score'] = .60 * ranking['historical_norm'] + .40 * ranking['content_score']
    observed = set(events.loc[events['school_name'].eq(school), 'action'])
    ranking['evidence_type'] = ['observed' if action in observed else 'predicted' for action in ranking.index]
    best = ranking.sort_values('hybrid_score', ascending=False).iloc[0]
    return {'status':'ok', 'school':school, 'action':best.name, 'hybrid_score':round(float(best['hybrid_score']),3), 'evidence_type':best['evidence_type'], 'hours':6}

@function_tool
def retrieve_brief_evidence(school: str, action: str, question: str) -> list[dict]:
    """Retrieve approved Lab 3 evidence for a recruiter-preparation question."""
    scores=cosine_similarity(vectorizer.transform([question]),corpus_matrix)[0]; picks=scores.argsort()[::-1][:3]
    return [{'source_id':corpus[i]['source_id'],'text':corpus[i]['text'][:500]} for i in picks]

# TODO: Attach the three approved tools already defined above. Do not add web search,
# messaging, scheduling, or any tool with an external side effect.
APPROVED_TOOLS = [rank_schools, recommend_action, retrieve_brief_evidence]

# TODO: Write the tool-agent contract. It must require ranking before choosing a
# school, action evidence before proposing an engagement, retrieval before local
# preparation facts, and escalation when evidence is unavailable.
TOOL_AGENT_INSTRUCTIONS = """Use approved tools before proposing a plan. Call ranking first, then an action recommendation for a ranked school, then retrieve approved preparation evidence. Preserve predicted/observed provenance. Do not exceed 16 hours, contact anyone, or make unsupported claims. If evidence is missing, say ESCALATE."""

tool_agent = Agent(name='Tool-Using Precision Recruiting Agent', model=MODEL, tools=APPROVED_TOOLS, instructions=TOOL_AGENT_INSTRUCTIONS)

async def run_tool_agent():
    return (await Runner.run(tool_agent, MISSION)).final_output

agent_output = await run_tool_agent()
display(Markdown(f'### Tool-agent result\n{agent_output}'))


In [ ]:
check('All three approved tools are attached', {tool.name for tool in APPROVED_TOOLS} == {'rank_schools', 'recommend_action', 'retrieve_brief_evidence'}, 'Attach only the three tools defined above.')
check('The tool agent is separate from the no-tool basic agent', tool_agent.name != bounded_basic_agent.name)


**Pause and return to the presentation.** Tool descriptions, state, schemas, and the observe–act loop are what turn the basic model interaction into a bounded agentic workflow.

# Section 3 — Evaluate and Red-Team the Agent

**Return to the presentation:** *Lab 4 Section 3 — evaluation.*

Evaluate the tools, the agent's trace, and the final report—not whether the prose merely sounds good.

In [ ]:
section_card('3', 'Validate and red-team the agent', 'Set concrete gates, then test whether a plan is escalated instead of improvised.')

# TODO: Set the mission limits used by validation.
MAX_PLAN_HOURS = 16
MIN_APPROVED_SOURCES = 2

def validate_plan(plan):
    problems = []
    if plan.get('hours', 0) > MAX_PLAN_HOURS: problems.append('field-hour budget exceeded')
    if len(plan.get('source_ids', [])) < MIN_APPROVED_SOURCES: problems.append('insufficient approved sources')
    return problems

happy_plan = {'hours': 6, 'source_ids': ['JHS_HANDBOOK_2026', 'JHS_CTE_GUIDE_2026']}
budget_failure = {'hours': 20, 'source_ids': happy_plan['source_ids']}
source_failure = {'hours': 6, 'source_ids': []}
evaluation = pd.DataFrame([{'case':'happy path','status':'proposed_for_review' if not validate_plan(happy_plan) else 'escalate'}, {'case':'budget failure','status':'proposed_for_review' if not validate_plan(budget_failure) else 'escalate'}, {'case':'source failure','status':'proposed_for_review' if not validate_plan(source_failure) else 'escalate'}])
evaluation


In [ ]:
check('Validation uses the 16-hour mission limit', MAX_PLAN_HOURS == 16)
check('Validation requires two approved sources', MIN_APPROVED_SOURCES == 2)
check('Only the supported happy path reaches review', evaluation['status'].tolist() == ['proposed_for_review', 'escalate', 'escalate'])


## Mission debrief

```text
Goal + instructions + state
        ↓
Basic agent: plausible but variable output
        ↓
Approved recommender and RAG tools
        ↓
Tool-calling loop: observe, continue, finish, or escalate
        ↓
Evaluate the trace and evidence
        ↓
Human-reviewable PDF report
```

An agent is valuable when it coordinates useful capabilities inside evidence, resource, and approval boundaries—not merely when it can produce an autonomous-sounding answer.